In [1]:
import gradio as gr
import pdfplumber
import json
import re

In [3]:
def extract_resume_text(file):
    """Extract text from a PDF resume."""
    with pdfplumber.open(file) as pdf:
        text = "\n".join([page.extract_text() for page in pdf.pages if page.extract_text()])
    return text

def parse_resume(text):
    """Parse resume text to extract relevant details (basic example)."""
    skills = re.findall(r'(?i)(Python|Machine Learning|Deep Learning|Data Science|AI Ethics)', text)
    education = "Master's" if "Master" in text else "Bachelor's" if "Bachelor" in text else "Unknown"
    experience = int(re.search(r'(\d+)\s+years?\s+experience', text).group(1)) if re.search(r'(\d+)\s+years?\s+experience', text) else 0
    work_type = "Remote" if "remote" in text.lower() else "Hybrid" if "hybrid" in text.lower() else "Onsite"
    location = "Any" if "willing to relocate" in text.lower() else "Unknown"
    
    return {
        "skills": skills,
        "education": education,
        "experience": experience,
        "work_type": work_type,
        "location": location
    }

def calculate_match_score(resume, job_role, job_description):
    """Calculate a weighted match score based on resume and job description."""
    weights = {
        "education": 0.2,
        "experience": 0.3,
        "technical_skills": 0.25,
        "soft_skills": 0.15,
        "work_type": 0.05,
        "location": 0.05
    }
    
    required_skills = job_description.get("required_skills", set())
    optional_skills = job_description.get("optional_skills", set())
    required_education = job_description.get("education", "Unknown")
    required_experience = job_description.get("experience", 0)
    required_work_type = job_description.get("work_type", "Any")
    required_location = job_description.get("location", "Any")
    
    if not required_skills:
        return 0.0, {}
    
    resume_skills = set(resume.get("skills", []))
    resume_education = resume.get("education", "Unknown")
    resume_experience = resume.get("experience", 0)
    resume_work_type = resume.get("work_type", "Any")
    resume_location = resume.get("location", "Any")
    
    matched_required_skills = resume_skills & required_skills
    matched_optional_skills = resume_skills & optional_skills
    missing_required_skills = required_skills - matched_required_skills
    missing_optional_skills = optional_skills - matched_optional_skills
    
    skill_score = (
        (len(matched_required_skills) * 1.0 + len(matched_optional_skills) * 0.5) /
        max(1, len(required_skills) + len(optional_skills))
    ) * 100
    
    experience_score = min(100, (resume_experience / required_experience) * 100) if required_experience else 100
    education_score = 100 if resume_education == required_education else 50 if resume_education else 0
    work_type_score = 100 if resume_work_type == required_work_type or required_work_type == "Any" else 75 if resume_work_type in ["Hybrid", "Remote"] else 0
    location_score = 100 if resume_location == required_location or required_location == "Any" else 50 if resume_location in required_location else 0
    
    match_percentage = (
        skill_score * weights["technical_skills"] +
        experience_score * weights["experience"] +
        education_score * weights["education"] +
        work_type_score * weights["work_type"] +
        location_score * weights["location"]
    )
    
    learning_links = [f"{skill}: Learning Resource Not Available" for skill in missing_required_skills]
    
    return round(match_percentage, 2), {
        "matched_required_skills": matched_required_skills,
        "matched_optional_skills": matched_optional_skills,
        "missing_required_skills": missing_required_skills,
        "missing_optional_skills": missing_optional_skills,
        "learning_links": learning_links,
        "education_match": education_score,
        "experience_match": experience_score,
        "work_type_match": work_type_score,
        "location_match": location_score
    }

def upload_resume(file):
    """Process uploaded resume (PDF) and compute match score."""
    resume_text = extract_resume_text(file.name)
    resume_data = parse_resume(resume_text)
    
    job_role = "AI Engineer"
    job_description = {
        "required_skills": {"Python", "Machine Learning", "Deep Learning"},
        "optional_skills": {"Data Science", "AI Ethics"},
        "education": "Master's",
        "experience": 3,
        "work_type": "Hybrid",
        "location": "Any"
    }
    
    match_score, details = calculate_match_score(resume_data, job_role, job_description)
    return f"Match Score: {match_score}%", details

demo = gr.Interface(
    fn=upload_resume,
    inputs=gr.File(type='filepath'),  # Corrected
    outputs=["text", "json"],
    title="AI-Driven Resume Screening Bot",
    description="Upload your resume (PDF format) to check job match score."
)


demo.launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [5]:
import gradio as gr
import json
from collections import Counter

# Dummy Job Role Dictionary (You can extend this)
JOB_ROLES = {
    "Data Scientist": {
        "skills": {"python", "ml", "ai", "data analysis"},
        "education": "Bachelor's in CS",
        "experience": 2,
        "work_type": "Remote",
        "location": "Any"
    },
    "Software Engineer": {
        "skills": {"java", "cloud", "database", "backend"},
        "education": "Bachelor's in CS",
        "experience": 3,
        "work_type": "On-site",
        "location": "New York"
    }
}

# Learning Resources (You can add more)
LEARNING_RESOURCES = {
    "python": "https://www.coursera.org/learn/python",
    "ml": "https://www.udacity.com/course/intro-to-ml--ud120",
    "ai": "https://www.deeplearning.ai/",
    "data analysis": "https://www.datacamp.com/courses/data-analysis",
    "java": "https://www.codecademy.com/learn/learn-java",
    "cloud": "https://www.aws.training/",
    "database": "https://www.udemy.com/course/sql-for-beginners/",
    "backend": "https://www.pluralsight.com/courses/web-development"
}

def analyze_job_description(job_description):
    """Analyze job description to determine weight distribution dynamically."""
    keywords = job_description.lower().split()
    keyword_counts = Counter(keywords)

    tech_keywords = {"python", "java", "ml", "ai", "cloud", "database", "software"}
    soft_skill_keywords = {"communication", "leadership", "teamwork", "problem-solving"}
    experience_keywords = {"years", "experience", "senior", "expert", "entry-level"}
    education_keywords = {"bachelor", "master", "phd", "degree"}

    tech_weight = sum(keyword_counts[word] for word in tech_keywords if word in keyword_counts)
    soft_skill_weight = sum(keyword_counts[word] for word in soft_skill_keywords if word in keyword_counts)
    experience_weight = sum(keyword_counts[word] for word in experience_keywords if word in keyword_counts)
    education_weight = sum(keyword_counts[word] for word in education_keywords if word in keyword_counts)

    total_weight = tech_weight + soft_skill_weight + experience_weight + education_weight
    if total_weight == 0:
        return {"technical_skills": 0.25, "soft_skills": 0.15, "experience": 0.3, "education": 0.2, "work_type": 0.05, "location": 0.05}

    return {
        "technical_skills": tech_weight / total_weight,
        "soft_skills": soft_skill_weight / total_weight,
        "experience": experience_weight / total_weight,
        "education": education_weight / total_weight,
        "work_type": 0.05,
        "location": 0.05
    }

def calculate_match_score(resume, job_role, job_description):
    """Calculate a weighted match score based on resume and job description."""
    weights = analyze_job_description(job_description)

    required_skills = JOB_ROLES.get(job_role, {}).get("skills", set())
    required_education = JOB_ROLES.get(job_role, {}).get("education", "")
    required_experience = JOB_ROLES.get(job_role, {}).get("experience", 0)
    required_work_type = JOB_ROLES.get(job_role, {}).get("work_type", "Any")
    required_location = JOB_ROLES.get(job_role, {}).get("location", "Any")

    if not required_skills:
        return 0.0, {}

    resume_skills = set(resume.get("skills", []))
    resume_education = resume.get("education", "")
    resume_experience = resume.get("experience", 0)
    resume_work_type = resume.get("work_type", "Any")
    resume_location = resume.get("location", "Any")

    matched_skills = resume_skills & required_skills
    missing_skills = required_skills - matched_skills
    skill_score = (len(matched_skills) / len(required_skills)) * 100 if required_skills else 0

    experience_score = min(100, (resume_experience / required_experience) * 100) if required_experience else 100
    education_score = 100 if resume_education == required_education else 0
    work_type_score = 100 if resume_work_type == required_work_type or required_work_type == "Any" else 0
    location_score = 100 if resume_location == required_location or required_location == "Any" else 0

    match_percentage = (
        skill_score * weights["technical_skills"] +
        experience_score * weights["experience"] +
        education_score * weights["education"] +
        work_type_score * weights["work_type"] +
        location_score * weights["location"]
    )

    learning_links = [f"{skill}: {LEARNING_RESOURCES.get(skill, 'No course available')}" for skill in missing_skills]

    return round(match_percentage, 2), {
        "Job Description": job_description,
        "Matched Skills": list(matched_skills),
        "Missing Skills": list(missing_skills),
        "Learning Links": learning_links,
        "Education Match": education_score,
        "Experience Match": experience_score,
        "Work Type Match": work_type_score,
        "Location Match": location_score
    }

def process_resume(file, job_role, job_description):
    """Process uploaded resume and calculate match score."""
    try:
        resume_data = json.load(file)
        match_score, details = calculate_match_score(resume_data, job_role, job_description)
        return f"Match Score: {match_score}%", details
    except Exception as e:
        return f"Error processing resume: {str(e)}", {}

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# 📄 Resume Job Match Scoring 🚀")
    
    with gr.Row():
        resume_file = gr.File(label="📂 Upload Resume (JSON Format)")
        job_role = gr.Dropdown(label="🎯 Select Job Role", choices=list(JOB_ROLES.keys()))
    
    job_description = gr.Textbox(label="📝 Paste Job Description Here", lines=5, placeholder="Paste job description...")

    submit_button = gr.Button("🔍 Calculate Match Score")

    output_text = gr.Textbox(label="✅ Match Score", interactive=False)
    output_details = gr.JSON(label="📊 Detailed Breakdown")

    submit_button.click(process_resume, inputs=[resume_file, job_role, job_description], outputs=[output_text, output_details])

demo.launch()


* Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


In [6]:
import fitz  # PyMuPDF for PDF extraction
import gradio as gr
import json

# Predefined job roles and descriptions
JOB_ROLES = {
    "Software Engineer": {
        "skills": {"Python", "Django", "REST API", "SQL"},
        "education": "Bachelor's in Computer Science",
        "experience": 2,
        "work_type": "Hybrid",
        "location": "New York",
        "weights": {"core": 0.5, "secondary": 0.3, "bonus": 0.2}
    }
}

LEARNING_RESOURCES = {
    "Python": "https://www.coursera.org/specializations/python",
    "Django": "https://www.udemy.com/course/python-and-django-full-stack-web-developer-bootcamp/"
}

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    doc = fitz.open(pdf_path)
    text = "\n".join([page.get_text("text") for page in doc])
    return text

def parse_resume(text):
    """Parses resume text into structured data (basic implementation)."""
    # In real-world use, NLP techniques like Named Entity Recognition (NER) should be used.
    return {
        "skills": {"Python", "Django", "REST API"},  # Extracted skills from text
        "education": "Bachelor's in Computer Science",
        "experience": 3,
        "work_type": "Remote",
        "location": "San Francisco"
    }

def calculate_match_score(resume, job_role):
    job_details = JOB_ROLES.get(job_role, {})
    required_skills = job_details.get("skills", set())
    weights = job_details.get("weights", {"core": 0.5, "secondary": 0.3, "bonus": 0.2})
    
    resume_skills = resume.get("skills", set())
    matched_skills = resume_skills & required_skills
    missing_skills = required_skills - matched_skills
    skill_score = (len(matched_skills) / len(required_skills)) * 100 if required_skills else 0
    
    experience_score = min(100, (resume["experience"] / job_details.get("experience", 1)) * 100)
    education_score = 100 if resume["education"] == job_details.get("education", "") else 0
    work_type_score = 100 if resume["work_type"] == job_details.get("work_type", "Any") else 0
    location_score = 100 if resume["location"] == job_details.get("location", "Any") else 0
    
    match_percentage = (
        skill_score * weights["core"] +
        experience_score * weights["secondary"] +
        education_score * weights["bonus"]
    )
    
    learning_links = [f"{skill}: {LEARNING_RESOURCES.get(skill, 'No course available')}" for skill in missing_skills]
    
    return round(match_percentage, 2), {
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "learning_links": learning_links,
        "education_match": education_score,
        "experience_match": experience_score,
        "work_type_match": work_type_score,
        "location_match": location_score
    }

def process_resume(pdf):
    text = extract_text_from_pdf(pdf.name)
    resume_data = parse_resume(text)
    score, details = calculate_match_score(resume_data, "Software Engineer")
    return f"Match Score: {score}%\nDetails: {json.dumps(details, indent=2)}"

giface = gr.Interface(
    fn=process_resume,
    inputs=gr.File(type="file"),
    outputs="text",
    title="Resume Matcher",
    description="Upload your PDF resume to check the job match score."
)

giface.launch()


ValueError: Invalid value for parameter `type`: file. Please choose from one of: ['filepath', 'binary']

In [8]:
!pip install PyPDF2


In [10]:
import gradio as gr
import PyPDF2
import json

def extract_text_from_pdf(pdf_file):
    text = ""
    with open(pdf_file, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

# Sample predefined job descriptions
JOB_ROLES = {
    "Software Engineer": {
        "skills": {"Python", "Django", "REST API", "SQL"},
        "education": "Bachelor's in Computer Science",
        "experience": 2,
        "work_type": "Hybrid",
        "location": "New York"
    },
    "Data Scientist": {
        "skills": {"Python", "Machine Learning", "Deep Learning", "Pandas", "SQL"},
        "education": "Master's in Data Science",
        "experience": 3,
        "work_type": "Remote",
        "location": "San Francisco"
    }
}

LEARNING_RESOURCES = {
    "Python": "https://www.coursera.org/specializations/python",
    "Django": "https://www.djangoproject.com/start/",
    "REST API": "https://restfulapi.net/",
    "Machine Learning": "https://www.udacity.com/course/intro-to-machine-learning--ud120"
}

def calculate_match_score(resume_text, job_role):
    """Calculate a weighted match score based on resume and job description."""
    
    if job_role not in JOB_ROLES:
        return "Job role not found."
    
    job_desc = JOB_ROLES[job_role]
    
    # Dynamically assign weights based on job role complexity
    skill_weight = 0.4 if len(job_desc["skills"]) > 5 else 0.3
    experience_weight = 0.3
    education_weight = 0.15
    work_type_weight = 0.05
    location_weight = 0.05
    
    resume_skills = set(resume_text.split())  # Extracting words as simple skill match
    matched_skills = resume_skills & job_desc["skills"]
    missing_skills = job_desc["skills"] - matched_skills
    
    skill_score = (len(matched_skills) / len(job_desc["skills"])) * 100 if job_desc["skills"] else 0
    experience_score = 100  # Assuming experience extraction (not implemented here)
    education_score = 100  # Assuming education extraction (not implemented here)
    work_type_score = 100 if "Hybrid" in resume_text else 0
    location_score = 100 if "New York" in resume_text else 0
    
    match_percentage = (
        skill_score * skill_weight +
        experience_score * experience_weight +
        education_score * education_weight +
        work_type_score * work_type_weight +
        location_score * location_weight
    )
    
    learning_links = [f"{skill}: {LEARNING_RESOURCES.get(skill, 'No course available')}" for skill in missing_skills]
    
    return round(match_percentage, 2), {
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "learning_links": learning_links
    }

def process_resume(pdf):
    resume_text = extract_text_from_pdf(pdf.name)
    job_role = "Software Engineer"  # Predefined job role (can be changed)
    score, details = calculate_match_score(resume_text, job_role)
    return f"Match Score: {score}%\n\nDetails:\n{json.dumps(details, indent=2)}"

iface = gr.Interface(
    fn=process_resume,
    inputs=gr.File(type="filepath", label="Upload Resume"),
    outputs="text",
    title="Resume Matcher",
    description="Upload your resume in PDF format and get a match score for a predefined job role."
)

iface.launch()


* Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\GAD\anaconda3\envs\llms\Lib\site-packages\gradio\queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\GAD\anaconda3\envs\llms\Lib\site-packages\gradio\route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\GAD\anaconda3\envs\llms\Lib\site-packages\gradio\blocks.py", line 2045, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\GAD\anaconda3\envs\llms\Lib\site-packages\gradio\blocks.py", line 1592, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\GAD\anaconda3\envs\llms\Lib\site-packages\anyio\to_thread.py", line 56, in run_sync
    return await get_asyn